# Day 22 - Quarantine and Reject Pattern

**Topic:** Quarantine and Reject Pattern  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกแยก records ที่ผ่าน validation ออกจาก bad records ด้วย `valid`, `quarantine`, และ `reject` pattern โดยไม่ drop ข้อมูลเสียทิ้งเงียบ ๆ

วันนี้เราจะต่อจากแนวคิด Data Quality Checks โดยฝึกสร้าง pattern สำหรับแยกข้อมูลที่พร้อมใช้ต่อ ออกจากข้อมูลที่ต้องรอตรวจสอบหรือแก้ไขก่อน เช่น missing key, invalid amount, invalid date, unknown customer และ invalid status ใน PySpark pipeline


## Goal of the day

1. แยกให้ออกว่า **valid**, **quarantine**, และ **reject** records ต่างกันอย่างไร
2. สร้าง `dq_issue_array` เพื่อเก็บ record-level issue tracking ได้
3. เขียน PySpark logic เพื่อ split good records และ bad records ออกจากกันอย่างเป็นระบบ
4. เขียน bad records ลง Lakehouse table เพื่อให้ตรวจสอบและ reprocess ได้
5. เข้าใจ production mindset ว่า bad records ไม่ควรถูก drop ทิ้งโดยไม่มี trace


## Why it matters in production

ใน production pipeline ข้อมูลเสียเกิดขึ้นได้ตลอด เช่น source ส่ง key หาย, amount ผิด, date parse ไม่ได้, status ไม่อยู่ใน expected domain หรือ reference master ไม่มี record ที่ match กัน

ถ้า pipeline เลือก `.filter()` เฉพาะข้อมูลดีแล้ว drop ข้อมูลเสียทิ้งทันที จะเกิดปัญหาใหญ่คือ:

- audit ย้อนหลังไม่ได้ว่า record ไหนเสียเพราะอะไร
- source/data owner ไม่เห็นปัญหาที่ต้องแก้
- reprocess ยาก เพราะ record เสียหายไปจาก flow แล้ว
- downstream อาจดูเหมือนข้อมูล clean แต่จริง ๆ มี data loss เงียบ ๆ
- incident investigation ทำได้ยาก เพราะไม่มี bad-record evidence

Production takeaway:

> Bad records should be separated, not silently dropped. Quarantine/Reject pattern ช่วยให้ pipeline เดินต่อได้แบบ traceable และยังรักษาหลักฐานของข้อมูลที่มีปัญหาไว้ตรวจสอบ


## Key concepts

| Concept | Meaning |
|---|---|
| Valid records | Records ที่ผ่าน validation และพร้อมส่งต่อ downstream |
| Quarantine records | Records ที่ไม่ผ่านบาง rules แต่ยังมี key/trace พอให้ review หรือแก้ไขแล้ว reprocess ได้ |
| Reject records | Records ที่มีปัญหาหนักหรือขาด field สำคัญจนไม่ควรส่งต่อ เช่น missing business key หรือ missing technical trace |
| Error reason | เหตุผลที่ record ถูก quarantine/reject เช่น `INVALID_AMOUNT`, `UNKNOWN_CUSTOMER_ID` |
| Rule severity | ระดับความรุนแรงของ issue เช่น blocking vs reviewable |
| Record-level tracking | การเก็บ issue ในระดับ record เพื่อรู้ว่าแต่ละแถว fail rule อะไร |
| Reprocessing | การนำ record ที่แก้ไขแล้วกลับมารัน validation ใหม่ |


## Concept explanation

### Quarantine vs Reject ต่างกันอย่างไร?

ในหลายองค์กร คำว่า `quarantine` และ `reject` อาจใช้แทนกันได้บางส่วน แต่สำหรับ learning lab วันนี้เราจะแยก mindset แบบนี้:

- **Quarantine** = record ยังมี key และ trace พอจะตรวจสอบได้ แต่มี business/data quality issue ที่ต้องแก้ก่อนส่งต่อ
- **Reject** = record ขาด field สำคัญหรือมี blocking issue จนไม่ควรถูกส่งต่อ downstream ในรอบนี้

ตัวอย่าง:

| Situation | Example | Pattern |
|---|---|---|
| amount เป็น 0 หรือติดลบ | `amount <= 0` | Quarantine |
| payment status ไม่อยู่ใน allowed values | `unknown` | Quarantine |
| customer ไม่มีใน reference master | `C999` | Quarantine |
| ไม่มี order id | `order_id is null` | Reject |
| ไม่มี customer id | `customer_id is null` | Reject |
| parse order date ไม่ได้ | `bad-date` | Reject |
| ไม่มี source record id | technical trace หาย | Reject |

> Definition จริงใน production ควรถูกตกลงร่วมกับ business/data owner แต่หลักคิดสำคัญคือ ต้องเก็บเหตุผลและหลักฐานของ bad records ไว้เสมอ

### ทำไมต้องมี error reason หลายค่าในหนึ่ง record?

หนึ่ง record อาจ fail มากกว่า 1 rule เช่น:

- missing customer id
- invalid amount
- invalid date

ถ้าเก็บแค่ `is_valid = false` จะรู้แค่ว่าไม่ผ่าน แต่ไม่รู้ว่าไม่ผ่านเพราะอะไร

ใน notebook นี้เราจะใช้ `dq_issue_array` เช่น:

```text
["INVALID_AMOUNT", "UNKNOWN_CUSTOMER_ID"]
```

เพื่อให้ record-level issue ชัดเจนและสามารถทำ summary ต่อได้ง่าย

### Pattern ของวันนี้

Flow หลักของวันนี้คือ:

1. Read / create candidate records
2. Standardize basic fields
3. Apply DQ rules
4. Build `dq_issue_array`
5. Classify เป็น `valid`, `quarantine`, หรือ `reject`
6. Write output tables แยกกัน
7. สร้าง issue summary เพื่อใช้ตรวจสอบภาพรวม


## Mermaid diagram: Quarantine and Reject Flow

```mermaid
flowchart LR
    A[Candidate records] --> B[Standardize fields]
    B --> C[Apply DQ rules]
    C --> D{Has issue?}
    D -- No --> E[Valid table]
    D -- Yes --> F{Blocking issue?}
    F -- Yes --> G[Reject table]
    F -- No --> H[Quarantine table]
    H --> I[Review and correction]
    I --> C
    G --> J[Escalate or exclude from downstream]
```

Key idea:

> Valid records ไปต่อได้ ส่วน bad records ต้องถูกแยกเก็บพร้อม reason เพื่อ review, correction และ reprocess ภายหลัง


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 3, Finished, Available, Finished, False)

## Create mock data

Dataset หลักคือ order/transaction-style records ที่มีทั้ง valid และ invalid cases:

- valid order
- invalid amount
- unknown customer
- missing order id
- missing customer id
- invalid date
- invalid payment status
- missing technical trace


In [2]:
orders_data = [
    ("SRC001", "O1001", "C001", "2026-05-01", 1200.00, "paid", "web", "BATCH_20260607"),
    ("SRC002", "O1002", "C002", "2026-05-01", 0.00, "paid", "web", "BATCH_20260607"),
    ("SRC003", "O1003", "C999", "2026-05-02", 850.00, "paid", "mobile", "BATCH_20260607"),
    ("SRC004", None, "C003", "2026-05-02", 300.00, "paid", "web", "BATCH_20260607"),
    ("SRC005", "O1005", None, "2026-05-03", 450.00, "paid", "branch", "BATCH_20260607"),
    ("SRC006", "O1006", "C004", "bad-date", 760.00, "paid", "web", "BATCH_20260607"),
    ("SRC007", "O1007", "C004", "2026-05-04", -50.00, "paid", "mobile", "BATCH_20260607"),
    ("SRC008", "O1008", "C001", "2026-05-04", 980.00, "unknown", "web", "BATCH_20260607"),
    ("SRC009", "O1009", "C003", "2026-05-05", 1100.00, "paid", "branch", "BATCH_20260607"),
    (None, "O1010", "C002", "2026-05-05", 500.00, "paid", "web", "BATCH_20260607"),
]

orders_schema = T.StructType([
    T.StructField("source_record_id", T.StringType(), True),
    T.StructField("order_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_date_raw", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("payment_status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("batch_id", T.StringType(), True),
])

df_orders_raw = spark.createDataFrame(orders_data, orders_schema)

df_orders_raw.show(truncate=False)
df_orders_raw.printSchema()

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 4, Finished, Available, Finished, False)

+----------------+--------+-----------+--------------+------+--------------+-------------+--------------+
|source_record_id|order_id|customer_id|order_date_raw|amount|payment_status|source_system|batch_id      |
+----------------+--------+-----------+--------------+------+--------------+-------------+--------------+
|SRC001          |O1001   |C001       |2026-05-01    |1200.0|paid          |web          |BATCH_20260607|
|SRC002          |O1002   |C002       |2026-05-01    |0.0   |paid          |web          |BATCH_20260607|
|SRC003          |O1003   |C999       |2026-05-02    |850.0 |paid          |mobile       |BATCH_20260607|
|SRC004          |NULL    |C003       |2026-05-02    |300.0 |paid          |web          |BATCH_20260607|
|SRC005          |O1005   |NULL       |2026-05-03    |450.0 |paid          |branch       |BATCH_20260607|
|SRC006          |O1006   |C004       |bad-date      |760.0 |paid          |web          |BATCH_20260607|
|SRC007          |O1007   |C004       |2026-05

### Create customer reference data

เราจะใช้ customer master เล็ก ๆ เพื่อจำลอง referential check ว่า `customer_id` ใน order ต้องมีอยู่ใน reference table


In [3]:
customer_master_data = [
    ("C001", "Alice", "active"),
    ("C002", "Bob", "active"),
    ("C003", "Charlie", "active"),
    ("C004", "Diana", "active"),
]

customer_master_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("customer_status", T.StringType(), True),
])

df_customer_master = spark.createDataFrame(customer_master_data, customer_master_schema)

df_customer_master.show(truncate=False)

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 5, Finished, Available, Finished, False)

+-----------+-------------+---------------+
|customer_id|customer_name|customer_status|
+-----------+-------------+---------------+
|C001       |Alice        |active         |
|C002       |Bob          |active         |
|C003       |Charlie      |active         |
|C004       |Diana        |active         |
+-----------+-------------+---------------+



## PySpark code examples

ใน section นี้เราจะสร้าง reusable classification logic แบบง่าย ๆ เพื่อให้ใช้ซ้ำได้ทั้ง main examples และ exercises


### Example 1: Build reusable classification logic

Function นี้จะทำ 4 เรื่องหลัก:

1. Standardize basic fields
2. Join กับ customer reference
3. สร้าง `dq_issue_array`
4. Classify record เป็น `valid`, `quarantine`, หรือ `reject`


In [4]:
allowed_payment_statuses = ["paid", "pending", "cancelled"]


def is_blank(column_name: str):
    return F.col(column_name).isNull() | (F.trim(F.col(column_name)) == "")


def classify_orders(df_input, df_customer_reference):
    """Classify order records into valid, quarantine, or reject."""
    df_standardized = (
        df_input
        .withColumn("source_record_id", F.trim(F.col("source_record_id")))
        .withColumn("order_id", F.trim(F.col("order_id")))
        .withColumn("customer_id", F.trim(F.col("customer_id")))
        .withColumn("order_date_raw", F.trim(F.col("order_date_raw")))
        .withColumn("order_date", F.to_date(F.col("order_date_raw"), "yyyy-MM-dd"))  # Uses the trimmed order_date_raw from the previous step.
        .withColumn("payment_status_normalized", F.lower(F.trim(F.col("payment_status"))))
        .withColumn("source_system_normalized", F.lower(F.trim(F.col("source_system"))))
    )

    df_customer_reference_clean = (
        df_customer_reference
        .select(F.trim(F.col("customer_id")).alias("ref_customer_id"))
        .dropDuplicates(["ref_customer_id"])
    )

    df_joined = (
        df_standardized.alias("o")
        .join(
            df_customer_reference_clean.alias("c"),
            F.col("o.customer_id") == F.col("c.ref_customer_id"),
            "left"
        )
        .select("o.*", F.col("c.ref_customer_id").alias("matched_customer_id"))
    )

    df_with_issues = (
        df_joined
        .withColumn(
            "dq_issue_candidates",
            F.array(
                F.when(is_blank("source_record_id"), F.lit("MISSING_SOURCE_RECORD_ID")),
                F.when(is_blank("order_id"), F.lit("MISSING_ORDER_ID")),
                F.when(is_blank("customer_id"), F.lit("MISSING_CUSTOMER_ID")),
                F.when(is_blank("order_date_raw") | F.col("order_date").isNull(), F.lit("INVALID_ORDER_DATE")),
                F.when(F.col("amount").isNull() | (F.col("amount") <= 0), F.lit("INVALID_AMOUNT")),
                F.when(
                    is_blank("payment_status") | (~F.col("payment_status_normalized").isin(allowed_payment_statuses)),
                    F.lit("INVALID_PAYMENT_STATUS")
                ),
                F.when(
                    (~is_blank("customer_id")) & F.col("matched_customer_id").isNull(),
                    F.lit("UNKNOWN_CUSTOMER_ID")
                ),
            )
        )
        # Remove null values from the issue array so only failed rules remain.
        .withColumn("dq_issue_array", F.expr("filter(dq_issue_candidates, x -> x is not null)"))
        .drop("dq_issue_candidates")
        .withColumn("issue_count", F.size(F.col("dq_issue_array")))
    )

    reject_condition = (
        F.array_contains(F.col("dq_issue_array"), "MISSING_SOURCE_RECORD_ID") |  # Check whether this issue exists inside the dq_issue_array.
        F.array_contains(F.col("dq_issue_array"), "MISSING_ORDER_ID") |
        F.array_contains(F.col("dq_issue_array"), "MISSING_CUSTOMER_ID") |
        F.array_contains(F.col("dq_issue_array"), "INVALID_ORDER_DATE")
    )

    df_classified = (
        df_with_issues
        .withColumn(
            "record_status",
            F.when(reject_condition, F.lit("reject"))
             .when(F.col("issue_count") > 0, F.lit("quarantine"))
             .otherwise(F.lit("valid"))
        )
        .withColumn(
            "issue_reason",
            F.when(F.col("issue_count") > 0, F.array_join(F.col("dq_issue_array"), "; "))  # Convert issue array into a readable semicolon-separated reason string.
             .otherwise(F.lit(None).cast("string"))
        )
        .withColumn("classified_at", F.current_timestamp())
    )

    return df_classified

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 6, Finished, Available, Finished, False)

### Example 2: Classify raw records

หลังจากสร้าง function แล้ว เราจะ apply กับ raw orders เพื่อดูว่าแต่ละ record ถูกจัดกลุ่มเป็นอะไร


In [5]:
df_orders_classified = classify_orders(df_orders_raw, df_customer_master)

classification_columns = [
    "source_record_id",
    "order_id",
    "customer_id",
    "order_date_raw",
    "order_date",
    "amount",
    "payment_status",
    "payment_status_normalized",
    "record_status",
    "dq_issue_array",
    "issue_reason",
]

df_orders_classified.select(classification_columns).show(truncate=False)

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 7, Finished, Available, Finished, False)

+----------------+--------+-----------+--------------+----------+------+--------------+-------------------------+-------------+--------------------------+------------------------+
|source_record_id|order_id|customer_id|order_date_raw|order_date|amount|payment_status|payment_status_normalized|record_status|dq_issue_array            |issue_reason            |
+----------------+--------+-----------+--------------+----------+------+--------------+-------------------------+-------------+--------------------------+------------------------+
|SRC001          |O1001   |C001       |2026-05-01    |2026-05-01|1200.0|paid          |paid                     |valid        |[]                        |NULL                    |
|SRC002          |O1002   |C002       |2026-05-01    |2026-05-01|0.0   |paid          |paid                     |quarantine   |[INVALID_AMOUNT]          |INVALID_AMOUNT          |
|SRC003          |O1003   |C999       |2026-05-02    |2026-05-02|850.0 |paid          |paid         

### Example 3: Check record count by status

ก่อนเขียน table ควรตรวจ summary ง่าย ๆ ก่อนว่า valid/quarantine/reject ออกมาเท่าไหร่

Expected result จาก mock data วันนี้:

- `valid` = 2 records
- `quarantine` = 4 records
- `reject` = 4 records


In [6]:
df_status_summary = (
    df_orders_classified
    .groupBy("record_status")
    .agg(F.count("*").alias("record_count"))
    .orderBy("record_status")
)

df_status_summary.show(truncate=False)

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 8, Finished, Available, Finished, False)

+-------------+------------+
|record_status|record_count|
+-------------+------------+
|quarantine   |4           |
|reject       |4           |
|valid        |2           |
+-------------+------------+



### Example 4: Split valid, quarantine, and reject records

หลัง classify แล้ว เราจะแยก DataFrame ออกเป็น 3 ชุด:

- `df_valid_orders` สำหรับ downstream
- `df_quarantine_orders` สำหรับ bad records ที่ review/reprocess ได้
- `df_reject_orders` สำหรับ records ที่มี blocking issue


In [7]:
df_valid_orders = (
    df_orders_classified
    .filter(F.col("record_status") == "valid")
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "amount",
        F.col("payment_status_normalized").alias("payment_status"),
        F.col("source_system_normalized").alias("source_system"),
        "batch_id",
        "source_record_id",
        "record_status",
        "classified_at",
    )
)

df_quarantine_orders = (
    df_orders_classified
    .filter(F.col("record_status") == "quarantine")
    .select(
        "source_record_id",
        "order_id",
        "customer_id",
        "order_date_raw",
        "order_date",
        "amount",
        "payment_status",
        "payment_status_normalized",
        "source_system",
        "batch_id",
        "record_status",
        "dq_issue_array",
        "issue_count",
        "issue_reason",
        "classified_at",
    )
)

df_reject_orders = (
    df_orders_classified
    .filter(F.col("record_status") == "reject")
    .select(
        "source_record_id",
        "order_id",
        "customer_id",
        "order_date_raw",
        "order_date",
        "amount",
        "payment_status",
        "payment_status_normalized",
        "source_system",
        "batch_id",
        "record_status",
        "dq_issue_array",
        "issue_count",
        "issue_reason",
        "classified_at",
    )
)

print("Valid records:", df_valid_orders.count())
print("Quarantine records:", df_quarantine_orders.count())
print("Reject records:", df_reject_orders.count())

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 9, Finished, Available, Finished, False)

Valid records: 2
Quarantine records: 4
Reject records: 4


### Example 5: Inspect separated outputs

การ inspect แยกทีละกลุ่มช่วยให้เห็นว่า records แบบไหนไป downstream ได้ และ records แบบไหนต้องถูกกักไว้ตรวจสอบก่อน


In [8]:
print("Valid orders")
df_valid_orders.show(truncate=False)

print("Quarantine orders")
df_quarantine_orders.select(
    "source_record_id",
    "order_id",
    "customer_id",
    "amount",
    "payment_status",
    "record_status",
    "issue_reason",
).show(truncate=False)

print("Reject orders")
df_reject_orders.select(
    "source_record_id",
    "order_id",
    "customer_id",
    "order_date_raw",
    "record_status",
    "issue_reason",
).show(truncate=False)

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 10, Finished, Available, Finished, False)

Valid orders
+--------+-----------+----------+------+--------------+-------------+--------------+----------------+-------------+--------------------------+
|order_id|customer_id|order_date|amount|payment_status|source_system|batch_id      |source_record_id|record_status|classified_at             |
+--------+-----------+----------+------+--------------+-------------+--------------+----------------+-------------+--------------------------+
|O1001   |C001       |2026-05-01|1200.0|paid          |web          |BATCH_20260607|SRC001          |valid        |2026-06-07 06:22:11.911952|
|O1009   |C003       |2026-05-05|1100.0|paid          |branch       |BATCH_20260607|SRC009          |valid        |2026-06-07 06:22:11.911952|
+--------+-----------+----------+------+--------------+-------------+--------------+----------------+-------------+--------------------------+

Quarantine orders
+----------------+--------+-----------+------+--------------+-------------+----------------------+
|source_rec

### Example 6: Write outputs to Fabric Lakehouse tables


In [9]:
valid_table_name = "day22_valid_orders"
quarantine_table_name = "day22_quarantine_orders"
reject_table_name = "day22_reject_orders"

(
    df_valid_orders
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(valid_table_name)
)

(
    df_quarantine_orders
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(quarantine_table_name)
)

(
    df_reject_orders
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(reject_table_name)
)

print("Written tables:")
print("-", valid_table_name)
print("-", quarantine_table_name)
print("-", reject_table_name)

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 11, Finished, Available, Finished, False)

Written tables:
- day22_valid_orders
- day22_quarantine_orders
- day22_reject_orders


### Example 7: Read quarantine table back for review

In [10]:
df_quarantine_from_table = spark.read.table(quarantine_table_name)

df_quarantine_from_table.select(
    "source_record_id",
    "order_id",
    "customer_id",
    "amount",
    "payment_status",
    "issue_reason",
    "classified_at",
).show(truncate=False)

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 12, Finished, Available, Finished, False)

+----------------+--------+-----------+------+--------------+----------------------+--------------------------+
|source_record_id|order_id|customer_id|amount|payment_status|issue_reason          |classified_at             |
+----------------+--------+-----------+------+--------------+----------------------+--------------------------+
|SRC002          |O1002   |C002       |0.0   |paid          |INVALID_AMOUNT        |2026-06-07 06:24:10.462962|
|SRC003          |O1003   |C999       |850.0 |paid          |UNKNOWN_CUSTOMER_ID   |2026-06-07 06:24:10.462962|
|SRC007          |O1007   |C004       |-50.0 |paid          |INVALID_AMOUNT        |2026-06-07 06:24:10.462962|
|SRC008          |O1008   |C001       |980.0 |unknown       |INVALID_PAYMENT_STATUS|2026-06-07 06:24:10.462962|
+----------------+--------+-----------+------+--------------+----------------------+--------------------------+



## Quick recap

| Question | Answer |
|---|---|
| Quarantine ใช้ทำอะไร? | เก็บ records ที่ fail DQ แต่ยัง review/แก้ไข/reprocess ได้ |
| Reject ใช้ทำอะไร? | เก็บ records ที่มี blocking issue จนไม่ควรส่งต่อ downstream ในรอบนี้ |
| ทำไมไม่ควร drop bad records ทิ้ง? | เพราะจะ audit ไม่ได้และเกิด silent data loss |
| `dq_issue_array` มีประโยชน์อย่างไร? | เก็บหลาย issue ใน record เดียว ทำให้ตรวจสอบและสรุปผลได้ง่าย |
| Valid table ควรมี record แบบไหน? | เฉพาะ records ที่ผ่าน blocking validation และพร้อมใช้ต่อ |
| Table quarantine/reject ควรเก็บอะไร? | raw values, normalized values บางส่วน, issue reason, batch id, source id และ timestamp |


## Exercises

### Exercise 1: Create issue-level summary table

สร้าง summary จาก `df_orders_classified` เพื่อดูว่าแต่ละ `record_status` มี issue code อะไรกี่ records

Requirements:

- ใช้ `explode(dq_issue_array)` เพื่อแตก issue array เป็นรายบรรทัด
- group by `record_status` และ `issue_code`
- count จำนวน affected records
- write เป็น Lakehouse table ชื่อ `day22_issue_summary`

Expected result:

- เห็น summary ว่า quarantine/reject fail rule อะไรบ้าง
- records ที่ valid ไม่ควรอยู่ใน summary นี้ เพราะไม่มี issue


In [11]:
df_issue_summary = (
    df_orders_classified
    .filter(F.size(F.col("dq_issue_array")) > 0)
    .select(
        "record_status",
        F.explode(F.col("dq_issue_array")).alias("issue_code")
    )
    .groupBy("record_status", "issue_code")
    .agg(F.count("*").alias("affected_records"))
    .orderBy("record_status", "issue_code")
)

df_issue_summary.show(truncate=False)

df_issue_summary.write.mode("overwrite").format("delta").saveAsTable("day22_issue_summary")

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 13, Finished, Available, Finished, False)

+-------------+------------------------+----------------+
|record_status|issue_code              |affected_records|
+-------------+------------------------+----------------+
|quarantine   |INVALID_AMOUNT          |2               |
|quarantine   |INVALID_PAYMENT_STATUS  |1               |
|quarantine   |UNKNOWN_CUSTOMER_ID     |1               |
|reject       |INVALID_ORDER_DATE      |1               |
|reject       |MISSING_CUSTOMER_ID     |1               |
|reject       |MISSING_ORDER_ID        |1               |
|reject       |MISSING_SOURCE_RECORD_ID|1               |
+-------------+------------------------+----------------+



### Exercise 2: Create downstream-ready DataFrame

สร้าง DataFrame ชื่อ `df_downstream_ready_orders` สำหรับส่งต่อ downstream transformation

Requirements:

- ใช้เฉพาะ records จาก `df_valid_orders`
- select เฉพาะ business columns หลัก
- เพิ่ม `ready_for_downstream = true`
- ใช้ `.count()` และ `.show()` เพื่อตรวจผล

Expected result:

- ได้เฉพาะ valid records เท่านั้น
- quarantine/reject records ไม่ถูกส่งต่อใน DataFrame นี้


In [12]:
df_downstream_ready_orders = (
    df_valid_orders
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "amount",
        "payment_status",
        "source_system",
    )
    .withColumn("ready_for_downstream", F.lit(True))
)

df_downstream_ready_orders.show(truncate=False)
print("Downstream-ready records:", df_downstream_ready_orders.count())

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 14, Finished, Available, Finished, False)

+--------+-----------+----------+------+--------------+-------------+--------------------+
|order_id|customer_id|order_date|amount|payment_status|source_system|ready_for_downstream|
+--------+-----------+----------+------+--------------+-------------+--------------------+
|O1001   |C001       |2026-05-01|1200.0|paid          |web          |true                |
|O1009   |C003       |2026-05-05|1100.0|paid          |branch       |true                |
+--------+-----------+----------+------+--------------+-------------+--------------------+

Downstream-ready records: 2


### Exercise 3: Simulate corrected quarantine records and reprocess

จำลองว่า data owner แก้ records ใน quarantine แล้วส่งกลับมาใหม่ จากนั้นใช้ `classify_orders()` function เดิมเพื่อตรวจว่า records กลายเป็น valid ได้หรือไม่

Requirements:

- สร้าง corrected records สำหรับ `O1002`, `O1003`, และ `O1008`
- แก้ amount, customer id หรือ payment status ให้ถูกต้อง
- run classification ใหม่
- ตรวจ count by `record_status`

Expected result:

- corrected records ควรกลายเป็น `valid`
- แสดงให้เห็นว่า quarantine pattern รองรับ reprocessing ได้


In [13]:
corrected_quarantine_data = [
    ("SRC002_RETRY", "O1002", "C002", "2026-05-01", 250.00, "paid", "web", "BATCH_20260607_RETRY"),
    ("SRC003_RETRY", "O1003", "C003", "2026-05-02", 850.00, "paid", "mobile", "BATCH_20260607_RETRY"),
    ("SRC008_RETRY", "O1008", "C001", "2026-05-04", 980.00, "paid", "web", "BATCH_20260607_RETRY"),
]

df_corrected_quarantine = spark.createDataFrame(corrected_quarantine_data, orders_schema)

df_corrected_classified = classify_orders(df_corrected_quarantine, df_customer_master)

df_corrected_classified.select(
    "source_record_id",
    "order_id",
    "customer_id",
    "amount",
    "payment_status_normalized",
    "record_status",
    "dq_issue_array",
    "issue_reason",
).show(truncate=False)

(
    df_corrected_classified
    .groupBy("record_status")
    .agg(F.count("*").alias("record_count"))
    .show(truncate=False)
)

StatementMeta(, f9e69dcf-6fd8-48a1-bab7-a1ebd84adcb5, 15, Finished, Available, Finished, False)

+----------------+--------+-----------+------+-------------------------+-------------+--------------+------------+
|source_record_id|order_id|customer_id|amount|payment_status_normalized|record_status|dq_issue_array|issue_reason|
+----------------+--------+-----------+------+-------------------------+-------------+--------------+------------+
|SRC002_RETRY    |O1002   |C002       |250.0 |paid                     |valid        |[]            |NULL        |
|SRC003_RETRY    |O1003   |C003       |850.0 |paid                     |valid        |[]            |NULL        |
|SRC008_RETRY    |O1008   |C001       |980.0 |paid                     |valid        |[]            |NULL        |
+----------------+--------+-----------+------+-------------------------+-------------+--------------+------------+

+-------------+------------+
|record_status|record_count|
+-------------+------------+
|valid        |3           |
+-------------+------------+



## Common mistakes

1. **Drop bad records ทิ้งหลัง filter**

   การ `.filter()` เฉพาะ records ที่ดีแล้วไม่เก็บ invalid records จะทำให้เกิด silent data loss และ audit ย้อนหลังไม่ได้

2. **เก็บแค่ `is_valid = false` แต่ไม่เก็บ reason**

   ถ้าไม่มี `issue_reason` หรือ `dq_issue_array` จะรู้แค่ว่า record เสีย แต่ไม่รู้ว่าเสียเพราะ rule ไหน

3. **ส่ง quarantine/reject records เข้า downstream โดยไม่ตั้งใจ**

   Downstream-ready DataFrame หรือ table ควรอ่านจาก valid set เท่านั้น ไม่ควร read จาก raw classified table โดยไม่ filter `record_status`

4. **ใช้ reject/quarantine definition ไม่ชัด**

   ใน production ต้องตกลงให้ชัดว่า issue แบบไหนคือ blocking, issue แบบไหน review/reprocess ได้ และใครเป็น owner ของการแก้ไข

5. **ไม่เก็บ trace columns**

   Quarantine/reject table ควรมี `source_record_id`, `batch_id`, source fields, issue reason และ timestamp เพื่อให้ตามกลับไปที่ source/load รอบนั้นได้

## Expected Output / Success Criteria

เมื่อจบ Day 22 ควรทำได้:

- อธิบายความต่างระหว่าง `valid`, `quarantine`, และ `reject` records ได้
- สร้าง `dq_issue_array` เพื่อเก็บหลาย issue ในหนึ่ง record ได้
- เขียน PySpark logic เพื่อ classify record-level DQ result ได้
- แยก DataFrame เป็น `df_valid_orders`, `df_quarantine_orders`, และ `df_reject_orders` ได้
- ตรวจ summary ได้ว่าแต่ละ issue code กระทบกี่ records
- อธิบายได้ว่าทำไม bad records ต้องถูกเก็บไว้ ไม่ใช่ถูก drop ทิ้งเงียบ ๆ
- จำลอง correction/reprocess ของ quarantine records ได้


## Final takeaway

Quarantine และ Reject Pattern คือ mindset สำคัญของ production data pipeline เพราะเป้าหมายไม่ใช่แค่ทำให้ downstream เห็นข้อมูล clean แต่ต้องทำให้ bad records ตรวจสอบย้อนหลังได้ด้วย

> Good records go forward. Bad records stay traceable.

สิ่งที่ควรจำจากวันนี้:

- อย่า drop bad records โดยไม่มี trace
- เก็บ issue reason ให้ละเอียดพอสำหรับ review
- แยก valid/quarantine/reject ให้ชัดเจน
- quarantine records ควร reprocess ได้หลัง correction
- reject definition ต้องชัด เพราะกระทบ data loss, audit และ downstream trust


## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day22_valid_orders")
# spark.sql("DROP TABLE IF EXISTS day22_quarantine_orders")
# spark.sql("DROP TABLE IF EXISTS day22_reject_orders")
# spark.sql("DROP TABLE IF EXISTS day22_issue_summary")